In [1]:
# ============================================================
# PaDiM training-ready slim dataset check
# ------------------------------------------------------------
# 목적:
#   - PaDiM 학습용 slim 폴더가 제대로 만들어졌는지 확인
#   - ct_hu.npy / pure_lung.npy / meta.json 존재 확인
#   - organ_exclusion.npy가 빠졌는지 확인
#   - patch CSV에서 불필요한 컬럼이 제거됐는지 확인
#   - train / val / test split, position count 확인
# ============================================================

from pathlib import Path
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm

PADIM_ROOT = Path(r"E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_padim_training_ready_v1")

VOLUME_ROOT = PADIM_ROOT / "volumes_npy"
PATCH_INDEX_ROOT = PADIM_ROOT / "patch_index_by_patient"
MANIFEST_DIR = PADIM_ROOT / "manifests"
CONFIG_DIR = PADIM_ROOT / "configs"
LOG_DIR = PADIM_ROOT / "logs"

manifest_path = MANIFEST_DIR / "patient_manifest.csv"
split_path = MANIFEST_DIR / "train_val_test_split.csv"
position_path = MANIFEST_DIR / "patch_position_counts.csv"
split_position_path = MANIFEST_DIR / "split_position_counts.csv"

print("========== ROOT CHECK ==========")
print("PADIM_ROOT exists:", PADIM_ROOT.exists())
print("VOLUME_ROOT exists:", VOLUME_ROOT.exists())
print("PATCH_INDEX_ROOT exists:", PATCH_INDEX_ROOT.exists())
print("MANIFEST_DIR exists:", MANIFEST_DIR.exists())
print("CONFIG_DIR exists:", CONFIG_DIR.exists())
print("LOG_DIR exists:", LOG_DIR.exists())

required_files = [
    manifest_path,
    split_path,
    position_path,
    split_position_path,
    CONFIG_DIR / "padim_training_ready_config.json",
]

print("\n========== REQUIRED FILE CHECK ==========")
for p in required_files:
    print(p.name, ":", p.exists(), "|", p)

if not manifest_path.exists():
    raise FileNotFoundError(f"patient_manifest.csv 없음: {manifest_path}")

manifest = pd.read_csv(manifest_path)

print("\n========== MANIFEST CHECK ==========")
print("manifest rows:", len(manifest))
display(manifest.head())

required_manifest_cols = [
    "patient_id",
    "safe_id",
    "patch_count",
    "volume_dir",
    "ct_hu_npy",
    "pure_lung_npy",
    "meta_json",
    "patch_csv",
]

missing_manifest_cols = [c for c in required_manifest_cols if c not in manifest.columns]
print("missing manifest cols:", missing_manifest_cols)

if len(missing_manifest_cols) > 0:
    raise RuntimeError(f"manifest 필수 컬럼 없음: {missing_manifest_cols}")

# ------------------------------------------------------------
# 환자별 파일 존재 확인
# ------------------------------------------------------------

file_check_rows = []

for _, row in tqdm(
    manifest.iterrows(),
    total=len(manifest),
    desc="Check patient files",
    ncols=100,
    ascii=True,
):
    patient_id = row["patient_id"]
    safe_id = row["safe_id"]

    ct_path = Path(str(row["ct_hu_npy"]))
    pure_path = Path(str(row["pure_lung_npy"]))
    meta_path = Path(str(row["meta_json"]))
    patch_csv = Path(str(row["patch_csv"]))

    patient_dir = Path(str(row["volume_dir"]))
    organ_path = patient_dir / "organ_exclusion.npy"

    file_check_rows.append({
        "patient_id": patient_id,
        "safe_id": safe_id,
        "ct_exists": ct_path.exists(),
        "pure_exists": pure_path.exists(),
        "meta_exists": meta_path.exists(),
        "patch_csv_exists": patch_csv.exists(),
        "organ_exclusion_exists": organ_path.exists(),
    })

file_check_df = pd.DataFrame(file_check_rows)

print("\n========== PATIENT FILE CHECK SUMMARY ==========")
display(file_check_df[[
    "ct_exists",
    "pure_exists",
    "meta_exists",
    "patch_csv_exists",
    "organ_exclusion_exists",
]].value_counts().reset_index(name="count"))

bad_files = file_check_df[
    (~file_check_df["ct_exists"])
    | (~file_check_df["pure_exists"])
    | (~file_check_df["meta_exists"])
    | (~file_check_df["patch_csv_exists"])
    | (file_check_df["organ_exclusion_exists"])
]

print("\nproblem patient rows:", len(bad_files))

if len(bad_files) > 0:
    display(bad_files.head(20))
else:
    print("환자별 필수 파일 OK / organ_exclusion.npy 제거 OK")

# ------------------------------------------------------------
# patch CSV 컬럼 확인
# ------------------------------------------------------------

EXPECTED_KEEP_COLS = [
    "patient_id",
    "safe_id",
    "label",

    "local_z",
    "y0",
    "x0",
    "y1",
    "x1",

    "patch_size",
    "patch_stride",

    "position_bin",
    "z_level",
    "z_ratio",
    "central_peripheral",
    "central_distance_ratio_mean",
    "left_right_metadata",

    "pure_lung_pixels",
    "organ_pixels",
    "pure_lung_patch_ratio",
    "organ_patch_ratio",
    "slice_pure_lung_ratio",
]

SHOULD_NOT_EXIST_PREFIXES = [
    "hu_",
    "hist_bin_",
]

SHOULD_NOT_EXIST_COLS = [
    "volume_dir",
    "ct_hu_npy",
    "pure_lung_npy",
    "organ_exclusion_npy",
    "volume_meta_json",
    "rel_volume_dir",
    "rel_ct_hu_npy",
    "rel_pure_lung_npy",
    "rel_organ_exclusion_npy",
    "rel_volume_meta_json",
    "ct_1mm_lung_range_nii",
    "pure_lung_lung_range_nii",
    "organ_exclusion_lung_range_nii",
]

patch_col_rows = []

# 전체 362개를 다 봐도 되지만, 처음에는 전체 확인해도 크게 오래 안 걸림
for _, row in tqdm(
    manifest.iterrows(),
    total=len(manifest),
    desc="Check patch CSV columns",
    ncols=100,
    ascii=True,
):
    patch_csv = Path(str(row["patch_csv"]))

    if not patch_csv.exists():
        patch_col_rows.append({
            "patient_id": row["patient_id"],
            "safe_id": row["safe_id"],
            "status": "patch_csv_missing",
            "missing_keep_cols": "",
            "bad_cols": "",
        })
        continue

    cols = pd.read_csv(patch_csv, nrows=0).columns.tolist()

    missing_keep_cols = [c for c in EXPECTED_KEEP_COLS if c not in cols]

    bad_cols = []

    for c in cols:
        if c in SHOULD_NOT_EXIST_COLS:
            bad_cols.append(c)

        for prefix in SHOULD_NOT_EXIST_PREFIXES:
            if c.startswith(prefix):
                bad_cols.append(c)

    bad_cols = sorted(set(bad_cols))

    patch_col_rows.append({
        "patient_id": row["patient_id"],
        "safe_id": row["safe_id"],
        "status": "ok" if len(missing_keep_cols) == 0 and len(bad_cols) == 0 else "check_needed",
        "missing_keep_cols": "|".join(missing_keep_cols),
        "bad_cols": "|".join(bad_cols),
        "col_count": len(cols),
    })

patch_col_df = pd.DataFrame(patch_col_rows)

print("\n========== PATCH CSV COLUMN CHECK ==========")
display(patch_col_df["status"].value_counts(dropna=False))

problem_cols = patch_col_df[patch_col_df["status"] != "ok"]

print("patch csv column problem rows:", len(problem_cols))

if len(problem_cols) > 0:
    display(problem_cols.head(20))
else:
    print("patch CSV 컬럼 OK")
    print("남은 컬럼 수:", patch_col_df["col_count"].unique())

# ------------------------------------------------------------
# split / position count 확인
# ------------------------------------------------------------

print("\n========== SPLIT CHECK ==========")

if split_path.exists():
    split_df = pd.read_csv(split_path)
    print("split rows:", len(split_df))
    display(split_df["split"].value_counts())
else:
    print("train_val_test_split.csv 없음")

print("\n========== POSITION COUNT CHECK ==========")

if position_path.exists():
    position_df = pd.read_csv(position_path)
    display(position_df)
else:
    print("patch_position_counts.csv 없음")

print("\n========== SPLIT POSITION COUNT CHECK ==========")

if split_position_path.exists():
    split_position_df = pd.read_csv(split_position_path)
    display(split_position_df)
else:
    print("split_position_counts.csv 없음")

# ------------------------------------------------------------
# 용량 확인
# ------------------------------------------------------------

def folder_size_gb(path: Path):
    total = 0
    for p in Path(path).rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total / (1024 ** 3)

print("\n========== SIZE CHECK ==========")
print("PADIM total:", f"{folder_size_gb(PADIM_ROOT):.2f} GB")
print("volumes_npy:", f"{folder_size_gb(VOLUME_ROOT):.2f} GB")
print("patch_index_by_patient:", f"{folder_size_gb(PATCH_INDEX_ROOT):.2f} GB")
print("manifests:", f"{folder_size_gb(MANIFEST_DIR):.2f} GB")
print("configs:", f"{folder_size_gb(CONFIG_DIR):.2f} GB")
print("logs:", f"{folder_size_gb(LOG_DIR):.2f} GB")

print("\n========== FINAL STRUCTURE CHECK RESULT ==========")

if len(bad_files) == 0 and len(problem_cols) == 0:
    print("전체 통과: PaDiM 학습용 slim 구조 정상")
else:
    print("부분 통과 또는 확인 필요")

========== ROOT CHECK ==========
PADIM_ROOT exists: True
VOLUME_ROOT exists: True
PATCH_INDEX_ROOT exists: True
MANIFEST_DIR exists: True
CONFIG_DIR exists: True
LOG_DIR exists: True

========== REQUIRED FILE CHECK ==========
patient_manifest.csv : True | E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_padim_training_ready_v1\manifests\patient_manifest.csv
train_val_test_split.csv : True | E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_padim_training_ready_v1\manifests\train_val_test_split.csv
patch_position_counts.csv : True | E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_padim_training_ready_v1\manifests\patch_position_counts.csv
split_position_counts.csv : True | E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_padim_training_ready_v1\manifests\split_position_counts.csv
padim_training_ready_config.json : True | E:\jyp\ct_data_2d_preprocessed\old\Normal_LUNA16_padim_training_ready_v1\configs\padim_training_ready_config.json

========== MANIFEST CHECK ==========
manifest rows: 3

,patient_id,safe_id,status,label,patch_count,volume_dir,ct_hu_npy,pure_lung_npy,meta_json,patch_csv,...,ct_copy_status,pure_copy_status,case_index,total_cases,elapsed_seconds,elapsed_readable,total_elapsed_seconds,total_elapsed_readable,estimated_remaining_seconds,estimated_remaining_readable
0,normal001,normal001__104e7cb873,success,normal,20993,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,...,hardlink,hardlink,1,362,0.47,0초,0.49,0초,176.40,2분 56초
1,normal002,normal002__d886c791fa,success,normal,22598,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,...,hardlink,hardlink,2,362,0.51,0초,1.04,1초,187.49,3분 7초
2,normal003,normal003__a8549e0718,success,normal,31194,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,...,hardlink,hardlink,3,362,0.68,0초,1.77,1초,212.09,3분 32초
3,normal004,normal004__9190565aec,success,normal,27154,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,...,hardlink,hardlink,4,362,0.59,0초,2.40,2초,215.00,3분 35초
4,normal005,normal005__41ac5e1f40,success,normal,32848,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_p...,...,hardlink,hardlink,5,362,0.71,0초,3.15,3초,225.22,3분 45초


missing manifest cols: []


Check patient files: 100%|######################################| 362/362 [00:00<00:00, 4389.77it/s]


========== PATIENT FILE CHECK SUMMARY ==========


...


========== POSITION COUNT CHECK ==========


,position_bin,patch_count
0,lower_central,829688
1,lower_peripheral,2214954
2,middle_central,1815279
3,middle_peripheral,4128678
4,upper_central,998059
5,upper_peripheral,2246813



========== SPLIT POSITION COUNT CHECK ==========


,split,position_bin,patch_count
0,test,lower_central,79463
1,test,lower_peripheral,212761
2,test,middle_central,169188
3,test,middle_peripheral,380140
4,test,upper_central,95180
5,test,upper_peripheral,210120
6,train,lower_central,663194
7,train,lower_peripheral,1775507
8,train,middle_central,1459641
9,train,middle_peripheral,3324054



========== SIZE CHECK ==========
PADIM total: 74.72 GB
volumes_npy: 71.46 GB
patch_index_by_patient: 3.26 GB
manifests: 0.00 GB
configs: 0.00 GB
logs: 0.00 GB

========== FINAL STRUCTURE CHECK RESULT ==========
부분 통과 또는 확인 필요


In [4]:
# ============================================================
# PaDiM sample patch loading check
# ------------------------------------------------------------
# 목적:
#   - ct_hu.npy / pure_lung.npy가 정상 로드되는지 확인
#   - patch CSV 좌표대로 32x32 patch가 잘리는지 확인
#   - pure_lung mask로 HU 계산 영역을 제한할 수 있는지 확인
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

PADIM_ROOT = Path(r"E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_padim_training_ready_v1")
MANIFEST_DIR = PADIM_ROOT / "manifests"
manifest = pd.read_csv(MANIFEST_DIR / "patient_manifest.csv")

# patch가 있는 환자 1명 선택
sample_patient = manifest[manifest["patch_count"].fillna(0).astype(int) > 0].iloc[0]

print("========== SAMPLE PATIENT ==========")
print("patient_id:", sample_patient["patient_id"])
print("safe_id:", sample_patient["safe_id"])
print("patch_count:", sample_patient["patch_count"])

ct_path = Path(str(sample_patient["ct_hu_npy"]))
pure_path = Path(str(sample_patient["pure_lung_npy"]))
meta_path = Path(str(sample_patient["meta_json"]))
patch_csv_path = Path(str(sample_patient["patch_csv"]))

ct = np.load(ct_path, mmap_mode="r")
pure = np.load(pure_path, mmap_mode="r")

with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("\n========== NPY CHECK ==========")
print("ct path:", ct_path)
print("pure path:", pure_path)
print("ct shape:", ct.shape, "dtype:", ct.dtype, "min/max:", int(ct.min()), int(ct.max()))
print("pure shape:", pure.shape, "dtype:", pure.dtype, "unique:", np.unique(pure))
print("meta shape_zyx:", meta.get("shape_zyx"))
print("meta spacing_xyz:", meta.get("spacing_xyz"))

if ct.shape != pure.shape:
    raise RuntimeError("ct_hu.npy와 pure_lung.npy shape가 다름")

patch_df = pd.read_csv(patch_csv_path)

print("\n========== PATCH CSV CHECK ==========")
print("patch csv:", patch_csv_path)
print("patch rows:", len(patch_df))
print("columns:", list(patch_df.columns))
display(patch_df.head())

# 첫 patch 하나 확인
p = patch_df.iloc[0]

z = int(p["local_z"])
y0 = int(p["y0"])
x0 = int(p["x0"])
y1 = int(p["y1"])
x1 = int(p["x1"])

ct_patch = ct[z, y0:y1, x0:x1]
pure_patch = pure[z, y0:y1, x0:x1]

hu_values_inside_pure = ct_patch[pure_patch > 0]

print("\n========== PATCH CROP CHECK ==========")
print("z, y0, x0, y1, x1:", z, y0, x0, y1, x1)
print("ct_patch shape:", ct_patch.shape, "dtype:", ct_patch.dtype)
print("pure_patch shape:", pure_patch.shape, "unique:", np.unique(pure_patch))
print("position_bin:", p["position_bin"])
print("pure_lung_patch_ratio in csv:", p["pure_lung_patch_ratio"])
print("pure_lung_patch_ratio actual:", float((pure_patch > 0).mean()))
print("pure HU pixel count:", len(hu_values_inside_pure))

if ct_patch.shape != (32, 32):
    raise RuntimeError(f"patch shape 이상함: {ct_patch.shape}")

if len(hu_values_inside_pure) == 0:
    raise RuntimeError("pure_lung 내부 HU 값이 없음")

print("\n========== PURE-LUNG HU FEATURE SAMPLE ==========")
print("hu_mean_inside_pure:", float(np.mean(hu_values_inside_pure)))
print("hu_std_inside_pure:", float(np.std(hu_values_inside_pure)))
print("hu_p05_inside_pure:", float(np.percentile(hu_values_inside_pure, 5)))
print("hu_p50_inside_pure:", float(np.percentile(hu_values_inside_pure, 50)))
print("hu_p95_inside_pure:", float(np.percentile(hu_values_inside_pure, 95)))

print("\n검증 통과: npy 로드, patch 좌표 crop, pure_lung 기준 HU 계산 가능")

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\jyp\\ct_data_2d_preprocessed\\Normal_LUNA16_padim_training_ready_v1\\manifests\\patient_manifest.csv'

In [ ]:
# ============================================================
# PaDiM patch visualization
# ------------------------------------------------------------
# 목적:
#   - 위치 그룹별 patch가 제대로 잘리는지 눈으로 확인
#   - CT slice 위에 pure_lung overlay와 patch box 표시
#   - 추출된 32x32 patch도 따로 표시
# ============================================================

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

PADIM_ROOT = Path(r"E:\jyp\ct_data_2d_preprocessed\Normal_LUNA16_padim_training_ready_v1")
MANIFEST_DIR = PADIM_ROOT / "manifests"

manifest = pd.read_csv(MANIFEST_DIR / "patient_manifest.csv")
split_df = pd.read_csv(MANIFEST_DIR / "train_val_test_split.csv")

# 보고 싶은 split 선택
TARGET_SPLIT = "train"
# TARGET_SPLIT = "val"
# TARGET_SPLIT = "test"

view_patients = split_df[split_df["split"] == TARGET_SPLIT].copy()

if len(view_patients) == 0:
    raise RuntimeError(f"{TARGET_SPLIT} split 환자가 없음")

# manifest 정보 붙이기
view_patients = view_patients.merge(
    manifest[
        [
            "patient_id",
            "safe_id",
            "patch_count",
            "ct_hu_npy",
            "pure_lung_npy",
            "meta_json",
            "patch_csv",
        ]
    ],
    on=["patient_id", "safe_id"],
    how="left",
    suffixes=("", "_manifest"),
)

# 랜덤 환자 여러 명에서 patch를 모음
random_state = 42
rng = random.Random(random_state)

# 위치 그룹별로 1개씩 보고 싶음
TARGET_POSITION_BINS = [
    "lower_central",
    "lower_peripheral",
    "middle_central",
    "middle_peripheral",
    "upper_central",
    "upper_peripheral",
]

sample_rows = []

patient_indices = list(view_patients.index)
rng.shuffle(patient_indices)

for pos in TARGET_POSITION_BINS:
    found = False

    for idx in patient_indices:
        prow = view_patients.loc[idx]
        patch_csv = Path(str(prow["patch_csv"]))

        if not patch_csv.exists():
            continue

        try:
            pdf = pd.read_csv(patch_csv)
        except Exception:
            continue

        cand = pdf[pdf["position_bin"] == pos].copy()

        if len(cand) == 0:
            continue

        row = cand.sample(1, random_state=random_state).iloc[0].to_dict()

        row["_ct_hu_npy"] = str(prow["ct_hu_npy"])
        row["_pure_lung_npy"] = str(prow["pure_lung_npy"])
        row["_meta_json"] = str(prow["meta_json"])
        row["_patient_id"] = str(prow["patient_id"])
        row["_safe_id"] = str(prow["safe_id"])

        sample_rows.append(row)
        found = True
        break

    if not found:
        print("[WARN] position_bin 샘플 없음:", pos)

sample_df = pd.DataFrame(sample_rows)

if len(sample_df) == 0:
    raise RuntimeError("시각화할 patch 샘플이 없음")

print("sample rows:", len(sample_df))
display(sample_df[[
    "_patient_id",
    "position_bin",
    "local_z",
    "y0",
    "x0",
    "y1",
    "x1",
    "pure_lung_patch_ratio",
    "organ_patch_ratio",
]])

def hu_to_uint8(arr, hu_min=-1000, hu_max=400):
    arr = np.clip(arr.astype(np.float32), hu_min, hu_max)
    arr = (arr - hu_min) / (hu_max - hu_min + 1e-8)
    arr = (arr * 255.0).astype(np.uint8)
    return arr

# 환자별 npy cache
ct_cache = {}
pure_cache = {}

for i, row in sample_df.iterrows():
    patient_id = row["_patient_id"]
    ct_path = row["_ct_hu_npy"]
    pure_path = row["_pure_lung_npy"]

    if ct_path not in ct_cache:
        ct_cache[ct_path] = np.load(ct_path, mmap_mode="r")

    if pure_path not in pure_cache:
        pure_cache[pure_path] = np.load(pure_path, mmap_mode="r")

    ct = ct_cache[ct_path]
    pure = pure_cache[pure_path]

    z = int(row["local_z"])
    y0 = int(row["y0"])
    x0 = int(row["x0"])
    y1 = int(row["y1"])
    x1 = int(row["x1"])

    ct_slice = ct[z]
    pure_slice = pure[z] > 0

    ct_patch = ct[z, y0:y1, x0:x1]
    pure_patch = pure[z, y0:y1, x0:x1] > 0

    hu_values_inside_pure = ct_patch[pure_patch]

    # -----------------------------
    # 1. 전체 slice + patch box
    # -----------------------------
    fig, ax = plt.subplots(figsize=(7, 7))

    ax.imshow(hu_to_uint8(ct_slice), cmap="gray")
    ax.imshow(pure_slice, cmap="Reds", alpha=0.22)

    rect = patches.Rectangle(
        (x0, y0),
        x1 - x0,
        y1 - y0,
        linewidth=2,
        edgecolor="yellow",
        facecolor="none",
    )
    ax.add_patch(rect)

    ax.set_title(
        f"{patient_id} | z={z}\n"
        f"{row['position_bin']} | "
        f"pure={row['pure_lung_patch_ratio']:.3f}, "
        f"organ={row['organ_patch_ratio']:.3f}"
    )
    ax.axis("off")
    plt.show()

    # -----------------------------
    # 2. 추출 patch + pure mask overlay
    # -----------------------------
    fig, ax = plt.subplots(figsize=(4, 4))

    ax.imshow(hu_to_uint8(ct_patch), cmap="gray")
    ax.imshow(pure_patch, cmap="Reds", alpha=0.25)

    if len(hu_values_inside_pure) > 0:
        title = (
            f"32x32 patch | {row['position_bin']}\n"
            f"HU mean={np.mean(hu_values_inside_pure):.1f}, "
            f"std={np.std(hu_values_inside_pure):.1f}"
        )
    else:
        title = f"32x32 patch | {row['position_bin']}"

    ax.set_title(title)
    ax.axis("off")
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def hu_to_uint8(arr, hu_min=-1000, hu_max=400):
    arr = np.clip(arr.astype(np.float32), hu_min, hu_max)
    arr = (arr - hu_min) / (hu_max - hu_min + 1e-8)
    return (arr * 255).astype(np.uint8)

# 이미 선택된 row 기준
z = int(row["local_z"])
y0 = int(row["y0"])
x0 = int(row["x0"])
y1 = int(row["y1"])
x1 = int(row["x1"])

ct_slice = ct[z]
pure_slice = pure[z] > 0

ct_patch = ct[z, y0:y1, x0:x1]
pure_patch = pure[z, y0:y1, x0:x1] > 0

print("csv pure ratio:", row["pure_lung_patch_ratio"])
print("actual pure ratio:", float(pure_patch.mean()))
print("patch coord:", z, y0, x0, y1, x1)

masked_pure = np.ma.masked_where(~pure_slice, pure_slice)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(hu_to_uint8(ct_slice), cmap="gray")
ax.imshow(masked_pure, cmap="Reds", alpha=0.35)

rect = patches.Rectangle(
    (x0, y0),
    x1 - x0,
    y1 - y0,
    linewidth=2,
    edgecolor="yellow",
    facecolor="none",
)
ax.add_patch(rect)

ax.set_title(
    f"MASKED overlay only where pure_lung=1\n"
    f"z={z}, pure={float(pure_patch.mean()):.3f}"
)
ax.axis("off")
plt.show()

plt.figure(figsize=(4, 4))
plt.imshow(hu_to_uint8(ct_patch), cmap="gray")
plt.imshow(np.ma.masked_where(~pure_patch, pure_patch), cmap="Reds", alpha=0.35)
plt.title("Patch + pure_lung mask")
plt.axis("off")
plt.show()